In [12]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch
from torch.optim import SGD, Adam
from transform import transform_log_scale, transform_cat
from sklearn.preprocessing import StandardScaler

In [9]:
df = df=pd.read_csv('Project_description_and_data/claims_train.csv')

In [19]:
#Preparing the data
df_scaled = transform_log_scale(df)
df_processed = transform_cat(df_scaled)
num_features = ['Exposure', 'VehPower', 'BonusMalus', 'VehAge_log', 'DrivAge_log', 'Density_log']
scaler = StandardScaler()
df_processed[num_features] = scaler.fit_transform(df_processed[num_features])
X = df_processed.drop(columns='ClaimNb')
y = df_processed['ClaimNb']

#splitting the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
class ModelDataset(Dataset):
    def __init__(self,x,y):
        self.x = x
        self.y = y
    def __getitem__(self,idx):
        return self.x[idx], self.y[idx]
    
    def __len__(self):
        return len(self.x)

In [21]:
X_train = X_train.astype("float32")
X_test = X_test.astype("float32")

In [23]:
X_train_np = X_train.to_numpy()
y_train_np = y_train.to_numpy().reshape(-1, 1)

X_test_np = X_test.to_numpy()
y_test_np = y_test.to_numpy().reshape(-1, 1)

ds = ModelDataset(torch.from_numpy(X_train_np),torch.from_numpy(y_train_np))
ds_test = ModelDataset(torch.from_numpy(X_test_np),torch.from_numpy(y_test_np))

train_loader = DataLoader(ds, batch_size=16, shuffle=True)
test_loader = DataLoader(ds_test, batch_size=16, shuffle=True)

In [24]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(64, 32)
        self.fc2 = nn.Linear(32, 10)
    def forward(self, x):
        #x = x.view(-1, 64)
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x
    
model = SimpleNN()

In [25]:
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.01)

for epoch in range(10):
    running_loss = 0.0
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch + 1}, Loss: {running_loss / len(train_loader)}")

RuntimeError: mat1 and mat2 shapes cannot be multiplied (16x48 and 64x32)